# MOOSE vs Eclipse – Croft primary depletion

This notebook uses the functions in `compare_with_eclipse.py` (keep it in the same folder), so it gives the same
figure as the command line, and lets you change anything about the plots.

1. Set the paths in the next cell.
2. Run all cells (**Run → Run All Cells**).
3. Edit the plotting cells to change colours, limits, panels, etc.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())          # folder with compare_with_eclipse.py
import numpy as np
import matplotlib.pyplot as plt
import compare_with_eclipse as cwe

# ---- paths: edit these -------------------------------------------------------------
ECLIPSE_H5 = r"CROFT_files/HM_MODEL.h5"      # Eclipse results written by Petrel
RUNS = {                                      # label -> the -o prefix given to em2ex2moose.py
    "MOOSE (em2ex mesh)": r"croft_em2ex",
    # "MOOSE (another run)": r"other_folder/croft_em2ex",
}
READ_FIELD_PRESSURE = True                    # reading <prefix>_exodus.e takes a minute or two

In [ ]:
runs = {label: cwe.read_moose(prefix, field_pressure=READ_FIELD_PRESSURE) for label, prefix in RUNS.items()}
units = next(iter(runs.values()))["meta"].get("units_in", "FIELD")
ecl = cwe.read_eclipse(ECLIPSE_H5, units=units)
print("Eclipse vectors:", [k for k in ecl if k not in ("t_days", "units")])
for label, d in runs.items():
    print(label, ":", [k for k in d if k not in ("t_days", "meta")])

## Standard comparison (same as the command line)

In [ ]:
fig, ax = cwe.plot_comparison(ecl, runs, title="Croft primary depletion: MOOSE vs Eclipse")
plt.show()

## Customising the plots

`plot_comparison` returns the figure and a dictionary of axes, keyed by panel:
`gip_sm3`, `gas_cum_sm3`, `gas_rate_sm3d`, `p_field_pa`, `p_well_block_pa`, `wip_change`.
Choose panels with `panels=[...]`, change line styles with `eclipse_style` / `run_styles`,
and edit any axis afterwards with ordinary matplotlib calls.

In [ ]:
fig = plt.figure(figsize=(12, 4.5))
fig, ax = cwe.plot_comparison(
    ecl, runs, fig=fig,
    panels=["gip_sm3", "p_field_pa", "p_well_block_pa"],
    eclipse_style=dict(color="black", lw=2, label="Eclipse (history match)"),
    run_styles={"MOOSE (em2ex mesh)": dict(color="tab:blue", ls="-", lw=1.5)},
)
ax["p_field_pa"].set_ylim(2250, 2900)
ax["p_well_block_pa"].set_ylim(1900, 2900)
for a in ax.values():
    a.set_xlim(0, 1330)
fig.tight_layout()
fig.savefig("croft_pressure_and_gip.png", dpi=150)
plt.show()

## Your own plot from the data

Every series is a numpy array in SI units: volumes in sm3 at surface conditions, rates in sm3/day,
pressures in Pa, time in days. The constants `cwe.SCF`, `cwe.PSI`, `cwe.STB` convert to field units.

In [ ]:
d = runs["MOOSE (em2ex mesh)"]
fig, axp = plt.subplots(figsize=(8, 4.5))
axp.plot(ecl["t_days"], ecl["p_field_pa"] / cwe.PSI, "k-", lw=2, label="Eclipse FPR")
axp.plot(d["t_field_days"], d["p_field_pa"] / cwe.PSI, "--", color="tab:red", lw=2, label="MOOSE, HCPV-weighted")
axp.plot(ecl["t_days"], ecl["p_well_block_pa"] / cwe.PSI, "k:", lw=1.5, label="Eclipse WBP")
axp.plot(d["t_days"], d["p_well_block_pa"] / cwe.PSI, ":", color="tab:red", lw=1.5, label="MOOSE well cell")
axp.set_xlabel("days since start"); axp.set_ylabel("pressure [psia]")
axp.grid(alpha=0.3); axp.legend()
plt.show()

## Numbers

In [ ]:
print(cwe.summary_table(ecl, runs, days=(0, 180, 365, 730, 1000, 1328.5)))

In [ ]:
# MOOSE minus Eclipse at chosen days (plot units)
diff = cwe.differences(ecl, runs["MOOSE (em2ex mesh)"], days=[180, 365, 730, 1328.5])
for k, v in diff.items():
    print("%-32s" % k, np.round(v, 3))